# Lab 05: Bias-Variance Tradeoff and Model Selection

Bias-variance tradeoff is a fundamental concept in machine learning that describes the "goodness" of a model/estimator.
Bias refers to the error introduced by approximating the true function occurring in a real-world problem with a model.
Variance refers to the error introduced by the model's sensitivity to fluctuations in the training data.
The decomposition is as follows:

$$
\text{MSE}(\mathbf{x}) = \text{Bias}(\mathbf{x})^2 + \text{Var}(\mathbf{x}) .
$$

In this lab, we will explore this concept in a real-world classification problem.

## The Iris Dataset

![Iris Dataset](iris.png)

Image obtained from <https://codesignal.com/learn/courses/intro-to-unsupervised-machine-learning/lessons/exploring-and-visualizing-the-iris-dataset>.

NOTES:
1) "Iris" is a genus name, and should start with an upper-case letter.
2) The labels for the "petal" point to the wrong structure. The small, veined, somewhat pointed parts are the petals.

In [ ]:
import numpy as np
np.random.seed(30009000)

from sklearn.datasets import load_iris

dataset = load_iris()
X, y = dataset.data, dataset.target

# X contains the flower measurements
print(X.shape)
# Y contains the class labels
print(y.shape)

Let's assume the data is noisy.
E.g., the researcher who collected the data made some errors measuring the flower structures.

We wouldn't do this on purpose, but it will make our example easier to see because points won't overlap.

In [ ]:
import numpy as np

noise_stddev = 0.5 #Controls amount of noise
X = X + np.random.normal(0, noise_stddev, size=X.shape)

As always, we will start by loading the dataset and splitting it.
In this lab, we introduce the third split of the dataset---the **validation set** which we will use later.

In [ ]:
from sklearn.model_selection import train_test_split

# 60-20-20 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=1)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=1)

print(
  f"Train size: {X_train.shape[0]}, val size: {X_val.shape[0]}, test size: {X_test.shape[0]}"
)

## K-Nearest Neighbors

We will use the K-Nearest Neighbors (KNN) algorithm to explore the bias-variance tradeoff in this classification problem.
The algorithm is simple: given the feature vector $x$ of a new data point, it looks at the $K$ nearest neighbors (nearest $x$) in the training set and uses the majority class among those neighbors as the prediction.

In [ ]:
from sklearn.metrics import pairwise_distances

def knn_predict(
  X_train: np.ndarray,
  y_train: np.ndarray,
  X_eval: np.ndarray,
  K: int
) -> np.ndarray:
  # Compute distances between test points and training points
  distances = pairwise_distances(X_eval, X_train)

  # Get the indices of the K nearest neighbors
  neighbors_idx = np.argsort(distances, axis=1)[:, :K]

  # Predict the majority class among the neighbors
  y_pred = []
  for idx in neighbors_idx:
      neighbor_labels = y_train[idx]
      majority_label = np.bincount(neighbor_labels).argmax()
      y_pred.append(majority_label)

  return np.array(y_pred)

# Example: Predictions for tests with 3 nearest neighbours
knn_predict(X_train,y_train,X_val,3)

We will use `scikit-learn`'s implementation of KNN nevertheless since it's optimized.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn3 = KNeighborsClassifier(n_neighbors=3)
knn3.fit(X_train, y_train)

# Should give same answer as our own implementation
knn3.predict(X_val)

Just as an example, let's evaluate the accuracy of our model on the training, validation, and test sets.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

K = 60

knn = KNeighborsClassifier(n_neighbors=K)
knn.fit(X_train, y_train)

y_pred_train = knn.predict(X_train)
y_pred_val = knn.predict(X_val)

print(f"Train accuracy: {accuracy_score(y_train, y_pred_train):.2f}")
print(f"Validation accuracy: {accuracy_score(y_val, y_pred_val):.2f}")

What if 60 isn't the best choice? Let's try 10.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

K = 10

knn = KNeighborsClassifier(n_neighbors=K)
knn.fit(X_train, y_train)

y_pred_train = knn.predict(X_train)
y_pred_val = knn.predict(X_val)

print(f"Train accuracy: {accuracy_score(y_train, y_pred_train):.2f}")
print(f"Validation accuracy: {accuracy_score(y_val, y_pred_val):.2f}")

## Model Selection

The KNN algorithm has a hyperparameter $K$ that controls the performance of the model.
Which value of $K$ should we choose?

Consider:
* What happens if we set K = N? (Number of data points in training set?)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import widgets


# Plot setup
plt.style.use("bmh")
plt.rcParams["font.family"] = ["Helvetica"]

# Which values of K we will consider
K_max = len(X_train)
K_range = range(1, K_max + 1)

# Lists to hold training and valication accuracies
train_accs = []
val_accs = []

for K in K_range:
  # Build classifier
  # NOTE: "fit" (at most) creates data structures to make lookup faster
  # it is not "fitting" the way we have described before -
  # kNN is a "lazy" algorithm
  knn = KNeighborsClassifier(n_neighbors=K)
  knn.fit(X_train, y_train)
  # Compute training accuracy
  train_accs.append(accuracy_score(y_train, knn.predict(X_train)))
  # Compute validation accuracy
  val_accs.append(accuracy_score(y_val, knn.predict(X_val)))

# Transform to array for convenience
train_accs = np.array(train_accs)
val_accs = np.array(val_accs)

print(train_accs.shape, val_accs.shape)

# Set range of data
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(
  np.linspace(x_min, x_max, 300),
  np.linspace(y_min, y_max, 300),
)

# Widget to interactively show different K.
@widgets.interact(K=widgets.IntSlider(value=1, min=1, max=K_max, step=1, description="K"))
def sweep_k(K: int) -> None:
  knn = KNeighborsClassifier(n_neighbors=K)
  knn.fit(X_train[:, :2], y_train)
  Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

  plt.contourf(xx, yy, Z, alpha=0.25, cmap="viridis")
  plt.scatter(
    X_train[:, 0], X_train[:, 1], c=y_train,
    cmap="viridis", edgecolors="k", s=30, label="Train",
  )
  plt.scatter(
    X_val[:, 0], X_val[:, 1], c=y_val,
    cmap="viridis", edgecolors="red", s=50, marker="*", label="Val",
  )
  plt.xlabel(dataset.feature_names[0])
  plt.ylabel(dataset.feature_names[1])
  plt.title(
    f"K = {K}\n"
    f"Train Acc = {train_accs[K-1]:.2f}, Val Acc = {val_accs[K-1]:.2f}"
  )
  plt.legend()
  plt.show()

### K-Fold Cross-Validation

We can make this model selection process more robust and automatic by using K-Fold Cross-Validation.
When using K-Fold Cross-Validation, we combine results from different ways of splitting to get a validation set.
All data "takes a turn" being the validation set, while the remaining is used for training.
This improves our estimate of validation error.

In [ ]:
from sklearn.model_selection import cross_val_score

X_train_val = np.concatenate([X_train, X_val], axis=0)
y_train_val = np.concatenate([y_train, y_val], axis=0)

val_accuracies = []

for K in K_range:
  model = KNeighborsClassifier(n_neighbors=K)
  # Choosing the splits and running the model is handled by this function
  scores = cross_val_score(model, X_train_val, y_train_val, cv=5)
  val_accuracies.append(scores.mean())

Let's visualize the results of K-Fold Cross-Validation.

In [ ]:
plt.plot(K_range, val_accuracies, label="Validation Accuracy")
plt.title("5-Fold Cross-Validation")
plt.xlabel("K")
plt.ylabel("Accuracy")
plt.xlim(1, K_max)
plt.ylim(0, 1)

Based on the validation accuracies, we can choose the best value of $K$ and evaluate the final model on the test set.

In [ ]:
best_K = K_range[np.argmax(val_accuracies)]

final_model = KNeighborsClassifier(n_neighbors=best_K)
final_model.fit(X_train_val, y_train_val)

test_acc = accuracy_score(y_test, final_model.predict(X_test))
print(f"Best K: {best_K}, Test Accuracy: {test_acc:.2f}")